In [ ]:
!pip install pandas joblib
!pip install scikit-learn==1.7.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 75.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np
import ast  # Library to safely evaluate string representations of Python literals
import warnings

warnings.filterwarnings('ignore')

# --- 1. Load the New, Unbiased Dataset ---
try:
    df = pd.read_csv('tmdb_5000_movies.csv')
    print("TMDB 5000 dataset loaded successfully.")

    # --- 2. Advanced Data Cleaning and Preparation ---
    print("Starting data cleaning and preparation...")

    # A. Handle missing financial data (treat 0 as missing)
    df['budget'].replace(0, np.nan, inplace=True)
    df['revenue'].replace(0, np.nan, inplace=True)
    df.dropna(subset=['budget', 'revenue'], inplace=True)
    print(f"Removed rows with missing financial data. Shape is now: {df.shape}")

    # B. Parse the JSON 'genres' column
    def parse_json_column(column_str):
        try:
            items = ast.literal_eval(column_str)
            names = [item['name'] for item in items]
            # Handle cases where genres might be empty
            if not names:
                return np.nan
            return ','.join(names)
        except (ValueError, SyntaxError, TypeError):
            return np.nan # Return NaN if parsing fails

    df['genres'] = df['genres'].apply(parse_json_column)

    # C. Handle release date
    # Some release dates might be malformed, so use errors='coerce'
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df.dropna(subset=['release_date', 'genres', 'runtime'], inplace=True)
    df['year'] = df['release_date'].dt.year

    # D. Rename columns to match the original script's expectations
    df.rename(columns={
        'revenue': 'box_office',
        'vote_average': 'rating',
        'runtime': 'run_time_minutes' # Runtime is already in minutes, just needs renaming
    }, inplace=True)

    print("Data cleaning complete.")

    # --- 3. Feature Selection and Splitting Data ---
    # Select the final features for the model
    features = ['year', 'rating', 'genres', 'run_time_minutes', 'budget']
    X = df[features]
    y = df['box_office']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Data split into training ({X_train.shape[0]} rows) and testing ({X_test.shape[0]} rows) sets.")

    # --- 4. Building the Model Pipeline (same as before) ---
    categorical_features = ['genres'] # Rating is now treated as a numerical feature
    numerical_features = ['year', 'rating', 'run_time_minutes', 'budget']

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])

    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ])
    print("Model pipeline created.")

    # --- 5. Train and Evaluate the New Model ---
    print("Training the new model...")
    model.fit(X_train, y_train)
    print("Model training complete.")

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n--- New Model Performance ---")
    print(f"Mean Squared Error (MSE): {mse:,.2f}")
    print(f"R-squared (R²): {r2:.2f}")

    # --- 6. Save the Newly Trained Model ---
    model_filename = 'movie_revenue_predictor.joblib'
    joblib.dump(model, model_filename)

    print(f"\nNew, improved model saved to '{model_filename}'.")
    print("You can now replace the old model file in your Flask app with this new one.")

except FileNotFoundError:
    print("Error: 'tmdb_5000_movies.csv' not found. Please ensure it's uploaded.")
except Exception as e:
    print(f"An error occurred: {e}")


TMDB 5000 dataset loaded successfully.
Starting data cleaning and preparation...
Removed rows with missing financial data. Shape is now: (3229, 20)
Data cleaning complete.
Data split into training (2582 rows) and testing (646 rows) sets.
Model pipeline created.
Training the new model...
Model training complete.

--- New Model Performance ---
Mean Squared Error (MSE): 20,013,921,597,324,100.00
R-squared (R²): 0.60

New, improved model saved to 'movie_revenue_predictor.joblib'.
You can now replace the old model file in your Flask app with this new one.
